In [35]:
import duckdb 
from pathlib import Path

In [36]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "warehouse.duckdb"

conn = duckdb.connect(str(DB_PATH), read_only=True)

In [ ]:
#1. daily prices to monthly / stg_prices 

conn.execute(""" 
    SELECT 
        date_trunc('month', date)   AS month,
        ticker,
        date,
        adj_close,
    FROM (
        SELECT
            date,
            ticker,
            adj_close,
            ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
        FROM stg_prices
             ) AS sub   
    WHERE sub.rn = 1
"""
).df()

,month,ticker,date,adj_close
0,2006-10-01,DBC,2006-10-31,19.177168
1,2008-01-01,DBC,2008-01-31,27.033495
2,2010-06-01,DBC,2010-06-30,18.236088
3,2010-10-01,DBC,2010-10-29,21.288118
4,2011-11-01,DBC,2011-11-30,23.367893
...,...,...,...,...
1554,2019-07-01,VNQ,2019-07-31,67.607376
1555,2020-06-01,VNQ,2020-06-30,61.934635
1556,2020-10-01,VNQ,2020-10-30,60.872723
1557,2022-05-01,VNQ,2022-05-31,82.766014


In [ ]:
# 

conn.execute(""" 
    SELECT 
        COUNT(ticker)
    FROM (
        SELECT
            date,
            ticker,
            adj_close,
            ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
        FROM stg_prices
             ) AS sub   
    WHERE sub.rn = 1
    GROUP BY ticker
"""
).df()

,count(ticker)
0,262
1,262
2,262
3,249
4,262
5,262


In [ ]:
# Nulls check

conn.execute("""
    WITH monthly_returns AS (
            SELECT 
        date_trunc('month', date)   AS month,
        ticker,
        date,
        adj_close,
    FROM (
        SELECT
            date,
            ticker,
            adj_close,
            ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
        FROM stg_prices
             ) AS sub   
    WHERE sub.rn = 1
    )
    SELECT COUNT(*) AS duplicated_keys
    FROM (
        SELECT month, ticker
        FROM monthly_returns
        GROUP BY month, ticker
        HAVING COUNT(*) > 1
    ) AS dup
""").fetchone()[0]

0

In [ ]:
# 2. monthly profitability

conn.execute("""
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close,
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nomial_return
            FROM monthly_prices
    )
    
    SELECT * FROM monthly_returns 
""").df()

,month,ticker,adj_close,nomial_return
0,2006-02-01,DBC,18.463675,NaN
1,2006-03-01,DBC,18.978979,0.027909
2,2006-04-01,DBC,20.310837,0.070175
3,2006-05-01,DBC,20.231558,-0.003903
4,2006-06-01,DBC,20.057150,-0.008621
...,...,...,...,...
1554,2026-06-01,VNQ,95.600838,0.016509
1555,2026-07-01,VNQ,98.099167,0.026133
1556,2026-08-01,VNQ,95.610756,-0.025366
1557,2026-09-01,VNQ,89.629997,-0.062553


In [ ]:
conn.execute("""
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close,
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    )
    
    SELECT COUNT(*) FROM monthly_returns WHERE nominal_return IS NULL
""").df()

,count_star()
0,6


In [ ]:
conn.execute("""
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close,
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    )
    
    SELECT  
        ticker,
        EXP(SUM(LN(1 + nominal_return))) - 1 AS chain_prof,
        arg_max(adj_close, month) / arg_min(adj_close, month) - 1 AS direct_prof
    FROM monthly_returns
    GROUP BY ticker
""").df()

,ticker,chain_prof,direct_prof
0,VNQ,3.582115,3.582115
1,TIP,0.997678,0.997678
2,DBC,0.820331,0.820331
3,GLD,7.426812,7.426812
4,SPY,8.501955,8.501955
5,TLT,0.818773,0.818773


In [43]:
#3. Monthly Inflation and yoy inflation

conn.execute("""
    WITH cpi_filled AS(
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    SELECT 
        month,
        cpi,
        cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
        cpi / LAG(cpi, 12) OVER (ORDER BY month) -1   AS yoy_inflation
    FROM cpi_filled
""").df()

,month,cpi,monthly_inflation,yoy_inflation
0,2005-01-01,191.600,NaN,NaN
1,2005-02-01,192.400,0.004175,NaN
2,2005-03-01,193.100,0.003638,NaN
3,2005-04-01,193.700,0.003107,NaN
4,2005-05-01,193.600,-0.000516,NaN
...,...,...,...,...
256,2026-05-01,333.979,0.004729,0.041666
257,2026-06-01,332.568,-0.004225,0.034635
258,2026-07-01,332.813,0.000737,0.033039
259,2026-08-01,334.131,0.003960,0.033530


In [44]:
#check how nulls are distributed

conn.execute("""
    WITH cpi_filled AS(
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    yoy_monthly_inflation AS (SELECT 
        month,
        cpi,
        cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
        cpi / LAG(cpi, 12) OVER (ORDER BY month) -1   AS yoy_inflation
    FROM cpi_filled)

    SELECT
        COUNT(*) - COUNT(monthly_inflation) AS monthly_nulls,
        COUNT(*) - COUNT(yoy_inflation)     AS yoy_nulls
    FROM yoy_monthly_inflation
    
""").df()

,monthly_nulls,yoy_nulls
0,1,12


In [45]:
conn.execute("""
    WITH cpi_filled AS(
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    yoy_monthly_inflation AS (SELECT 
        month,
        cpi,
        cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
        cpi / LAG(cpi, 12) OVER (ORDER BY month) -1   AS yoy_inflation
    FROM cpi_filled)

    SELECT
        month,
        monthly_inflation,
        yoy_inflation
    FROM yoy_monthly_inflation
    WHERE monthly_inflation IS NULL OR yoy_inflation IS NULL
    
""").df()

,month,monthly_inflation,yoy_inflation
0,2005-01-01,NaN,NaN
1,2005-02-01,0.004175,NaN
2,2005-03-01,0.003638,NaN
3,2005-04-01,0.003107,NaN
4,2005-05-01,-0.000516,NaN
5,2005-06-01,0.000517,NaN
6,2005-07-01,0.006195,NaN
7,2005-08-01,0.006157,NaN
8,2005-09-01,0.013768,NaN
9,2005-10-01,0.001509,NaN


In [46]:
#Searching for   extreme cases

conn.execute("""
    WITH cpi_filled AS(
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    yoy_monthly_inflation AS (SELECT 
        month,
        cpi,
        cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
        cpi / LAG(cpi, 12) OVER (ORDER BY month) -1   AS yoy_inflation
    FROM cpi_filled)

    SELECT
        month,
        monthly_inflation,
        yoy_inflation
    FROM yoy_monthly_inflation
    WHERE yoy_inflation IS NOT NULL
    ORDER BY yoy_inflation DESC
""").df()

,month,monthly_inflation,yoy_inflation
0,2022-06-01,0.012561,0.089794
1,2022-03-01,0.011156,0.085722
2,2022-05-01,0.009485,0.085382
3,2022-07-01,-0.000149,0.084626
4,2022-04-01,0.003083,0.082318
...,...,...,...
244,2009-05-01,0.001471,-0.010158
245,2009-06-01,0.008300,-0.012292
246,2009-09-01,0.001931,-0.013779
247,2009-08-01,0.003348,-0.014838


In [47]:
conn.execute("""
    WITH inflation AS (
        SELECT 
            date                                            AS month,
            value                                           AS cpi,
            value / LAG(value) OVER (ORDER BY date) -1      AS monthly_inflation,
            value / LAG(value,12) OVER (ORDER BY date) -1   AS yoy_inflation
        FROM stg_fred
        WHERE series_id = 'CPIAUCSL' 
    ),

    SELECT
        month,
        monthly_inflation,
        yoy_inflation
    FROM inflation
    WHERE yoy_inflation IS NOT NULL
    ORDER BY monthly_inflation DESC
""").df()

,month,monthly_inflation,yoy_inflation
0,2022-06-01,0.012561,0.089794
1,2022-03-01,0.011156,0.085722
2,2008-06-01,0.010478,0.049360
3,2021-10-01,0.009638,0.062350
4,2022-05-01,0.009485,0.085382
...,...,...,...
243,2020-04-01,-0.007920,0.003130
244,2008-12-01,-0.008234,-0.000222
245,2008-10-01,-0.008598,0.037311
246,2008-11-01,-0.017705,0.010999


In [48]:
# 4. cash prof /// TB3MS is an interest rate not a price! So we have to convert it to a price.
#We have to use the avg from the previuos month. Look-ahead bias.

conn.execute("""
    SELECT 
        date        AS month,
        'CASH'      AS ticker,
        POWER(1 + LAG(value) OVER (ORDER BY month) / 100, 1.0 /12)  - 1     AS nominal_return
    FROM stg_fred
    WHERE series_id = 'TB3MS'
""").df()

,month,ticker,nominal_return
0,2005-01-01,CASH,NaN
1,2005-02-01,CASH,0.001921
2,2005-03-01,CASH,0.002092
3,2005-04-01,CASH,0.002255
4,2005-05-01,CASH,0.002288
...,...,...,...
256,2026-05-01,CASH,0.002960
257,2026-06-01,CASH,0.002952
258,2026-07-01,CASH,0.003000
259,2026-08-01,CASH,0.003056


In [49]:
#4b -- checking nulls in cash

conn.execute("""
    WITH cash AS (
    SELECT 
        date        AS month,
        'CASH'      AS ticker,
        LAG(value) OVER (ORDER BY month)                    AS last_month_value,
        POWER(1 + last_month_value / 100, 1.0 /12)  - 1     AS nominal_return,
    FROM stg_fred
    WHERE series_id = 'TB3MS'
    ),

    SELECT COUNT(*)
    FROM cash
    WHERE nominal_return IS NULL

""").df()

,count_star()
0,1


In [50]:
#4c -- avg between 2009 and 2015 should be near 0

conn.execute("""
    WITH cash AS (
    SELECT 
        date        AS month,
        'CASH'      AS ticker,
        LAG(value) OVER (ORDER BY month)                    AS last_month_value,
        POWER(1 + last_month_value / 100, 1.0 /12)  - 1     AS nominal_return,
    FROM stg_fred
    WHERE series_id = 'TB3MS'
    ),

    SELECT AVG(nominal_return)
    FROM cash
    WHERE month BETWEEN '2009-01-01' AND '2015-01-01'

""").df()

,avg(nominal_return)
0,0.000071


In [51]:
#4d -- values of last 5 years

conn.execute("""
    WITH cash AS (
    SELECT 
        date        AS month,
        'CASH'      AS ticker,
        LAG(value) OVER (ORDER BY month)                    AS last_month_value,
        POWER(1 + last_month_value / 100, 1.0 /12)  - 1     AS nominal_return,
    FROM stg_fred
    WHERE series_id = 'TB3MS'
    ),

    SELECT date_trunc('year', month) as year, AVG(nominal_return)
    FROM cash
    WHERE month BETWEEN '2020-01-01' AND '2026-01-01'
    GROUP BY year
    ORDER BY year DESC

""").df()

,year,avg(nominal_return)
0,2026-01-01,0.002944
1,2025-01-01,0.003373
2,2024-01-01,0.004112
3,2023-01-01,0.004062
4,2022-01-01,0.001376
5,2021-01-01,0.000039
6,2020-01-01,0.000403


In [ ]:
#5 get every consult in 1 table 

conn.execute(""" 
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    ),

    cpi_filled AS (
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    inflation AS (
        SELECT 
            cpi,
            cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
            cpi / LAG(cpi,12) OVER (ORDER BY month) -1   AS yoy_inflation
        FROM cpi_filled 
    ),

    cash_returns AS (
        SELECT 
            date        AS month,
            'CASH'      AS ticker,
            POWER(1 + LAG(value) OVER (ORDER BY month) / 100, 1.0 /12)  - 1     AS nominal_return
        FROM stg_fred
        WHERE series_id = 'TB3MS'
    ),

    all_returns AS (
        SELECT month, ticker, nominal_return FROM monthly_returns
        UNION ALL
        SELECT month, ticker, nominal_return FROM cash_returns
    ),

    real_returns AS (
        SELECT
            a.month,
            ticker,
            nominal_return,
            monthly_inflation,
            (1 + nominal_return) / (1 + monthly_inflation) - 1   AS real_returns
        FROM all_returns a
        INNER JOIN inflation b
        ON a.month = b.month
        WHERE a.month >= DATE '2006-03-01'
        AND b.month <= (SELECT MAX(month) FROM inflation WHERE monthly_inflation IS NOT NULL)
    )

    SELECT * FROM real_returns
""").df()

BinderException: Binder Error: Values list "b" does not have a column named "month"

LINE 72:         ON a.month = b.month
                                    ^

In [ ]:
#5 rows x ticker

conn.execute(""" 
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    ),

    cpi_filled AS (
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    inflation AS (
        SELECT 
            month,
            cpi,
            cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
            cpi / LAG(cpi,12) OVER (ORDER BY month) -1   AS yoy_inflation
        FROM cpi_filled 
    ),

    cash_returns AS (
        SELECT 
            date        AS month,
            'CASH'      AS ticker,
            POWER(1 + LAG(value) OVER (ORDER BY month) / 100, 1.0 /12)  - 1     AS nominal_return
        FROM stg_fred
        WHERE series_id = 'TB3MS'
    ),

    all_returns AS (
        SELECT month, ticker, nominal_return FROM monthly_returns
        UNION ALL
        SELECT month, ticker, nominal_return FROM cash_returns
    ),

    real_returns AS (
        SELECT
            a.month,
            ticker,
            nominal_return,
            monthly_inflation,
            (1 + nominal_return) / (1 + monthly_inflation) - 1   AS real_returns
        FROM all_returns a
        INNER JOIN inflation b
        ON a.month = b.month
        WHERE a.month >= DATE '2006-03-01'
        AND a.month <= (SELECT MAX(month) FROM inflation WHERE monthly_inflation IS NOT NULL)
    )

    SELECT COUNT(*) FROM real_returns GROUP BY ticker
""").df()

,count_star()
0,247
1,247
2,247
3,247
4,247
5,247
6,247


In [ ]:
#5 nulls in real_returns

conn.execute(""" 
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    ),

    cpi_filled AS (
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    inflation AS (
        SELECT 
            month,
            cpi,
            cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
            cpi / LAG(cpi,12) OVER (ORDER BY month) -1   AS yoy_inflation
        FROM cpi_filled 
    ),

    cash_returns AS (
        SELECT 
            date        AS month,
            'CASH'      AS ticker,
            POWER(1 + LAG(value) OVER (ORDER BY month) / 100, 1.0 /12)  - 1     AS nominal_return
        FROM stg_fred
        WHERE series_id = 'TB3MS'
    ),

    all_returns AS (
        SELECT month, ticker, nominal_return FROM monthly_returns
        UNION ALL
        SELECT month, ticker, nominal_return FROM cash_returns
    ),

    real_returns AS (
        SELECT
            a.month,
            ticker,
            nominal_return,
            monthly_inflation,
            (1 + nominal_return) / (1 + monthly_inflation) - 1   AS real_returns
        FROM all_returns a
        INNER JOIN inflation b
        ON a.month = b.month
        WHERE a.month >= DATE '2006-03-01'
        AND a.month <= (SELECT MAX(month) FROM inflation WHERE monthly_inflation IS NOT NULL)
    )

    SELECT COUNT(*) FROM real_returns WHERE real_returns IS NULL
""").df()

,count_star()
0,0


In [ ]:
#5 Looking for negative returns

conn.execute(""" 
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    ),

    cpi_filled AS (
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    inflation AS (
        SELECT 
            month,
            cpi,
            cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
            cpi / LAG(cpi,12) OVER (ORDER BY month) -1   AS yoy_inflation
        FROM cpi_filled 
    ),

    cash_returns AS (
        SELECT 
            date        AS month,
            'CASH'      AS ticker,
            POWER(1 + LAG(value) OVER (ORDER BY month) / 100, 1.0 /12)  - 1     AS nominal_return
        FROM stg_fred
        WHERE series_id = 'TB3MS'
    ),

    all_returns AS (
        SELECT month, ticker, nominal_return FROM monthly_returns
        UNION ALL
        SELECT month, ticker, nominal_return FROM cash_returns
    ),

    real_returns AS (
        SELECT
            a.month,
            ticker,
            nominal_return,
            monthly_inflation,
            (1 + nominal_return) / (1 + monthly_inflation) - 1   AS real_returns
        FROM all_returns a
        INNER JOIN inflation b
        ON a.month = b.month
        WHERE a.month >= DATE '2006-03-01'
        AND a.month <= (SELECT MAX(month) FROM inflation WHERE monthly_inflation IS NOT NULL)
    )

    SELECT date_trunc('year', month) AS year, ticker, avg(real_returns) FROM real_returns WHERE real_returns < 0 GROUP BY ticker, year
""").df()

,year,ticker,avg(real_returns)
0,2023-01-01,TLT,-0.035749
1,2012-01-01,TIP,-0.008153
2,2013-01-01,TIP,-0.020487
3,2012-01-01,GLD,-0.024395
4,2013-01-01,GLD,-0.051254
...,...,...,...
142,2014-01-01,CASH,-0.001444
143,2017-01-01,CASH,-0.002207
144,2021-01-01,CASH,-0.005716
145,2022-01-01,CASH,-0.006454


In [ ]:
#6

conn.execute(""" 
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    ),

    cpi_filled AS (
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    inflation AS (
        SELECT 
            month,
            cpi,
            cpi_is_imputed,
            cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
            cpi / LAG(cpi,12) OVER (ORDER BY month) -1   AS yoy_inflation
        FROM cpi_filled 
    ),

    cash_returns AS (
        SELECT 
            date        AS month,
            'CASH'      AS ticker,
            POWER(1 + LAG(value) OVER (ORDER BY month) / 100, 1.0 /12)  - 1     AS nominal_return
        FROM stg_fred
        WHERE series_id = 'TB3MS'
    ),

    all_returns AS (
        SELECT month, ticker, nominal_return FROM monthly_returns
        UNION ALL
        SELECT month, ticker, nominal_return FROM cash_returns
    ),

    real_returns AS (
        SELECT
            a.month,
            ticker,
            nominal_return,
            monthly_inflation,
            yoy_inflation,
            (1 + nominal_return) / (1 + monthly_inflation) - 1   AS real_return
        FROM all_returns a
        INNER JOIN inflation b
        ON a.month = b.month
        WHERE a.month >= DATE '2006-03-01'
        AND a.month <= (SELECT MAX(month) FROM inflation WHERE monthly_inflation IS NOT NULL)
    ),

    wealth AS (
        SELECT
            month,
            ticker,
            nominal_return,
            real_return,
            EXP(SUM(LN(1 + nominal_return)) OVER (
                PARTITION BY ticker
                ORDER BY month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )) AS nominal_wealth,
            EXP(SUM(LN(1 + real_return)) OVER (
                PARTITION BY ticker
                ORDER BY month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )) AS real_wealth
        FROM real_returns
    ),

    drawdowns AS (
        SELECT
            month,
            ticker,
            nominal_return,
            real_return,
            nominal_wealth,
            real_wealth,
            real_wealth / GREATEST(1.0, MAX(real_wealth) OVER (
                PARTITION BY ticker
                ORDER BY month
            )) - 1 AS real_drawdown
        FROM wealth
    )

    SELECT
        ticker,
        arg_max(real_wealth, month)                                     AS final_real_wealth,
        POWER(arg_max(real_wealth, month), 1.0 / (COUNT(*) / 12.0)) - 1 AS real_cagr,
        MIN(real_drawdown)                                              AS max_real_drawdown,
        arg_min(month, real_drawdown)                                   AS max_drawdown_month
    FROM drawdowns
    GROUP BY ticker
    ORDER BY real_cagr DESC
   """ ).df()

,ticker,final_real_wealth,real_cagr,max_real_drawdown,max_drawdown_month
0,SPY,5.148195,0.082865,-0.515981,2009-02-01
1,GLD,4.027290,0.070024,-0.457087,2015-12-01
2,VNQ,1.933203,0.032543,-0.696824,2009-02-01
3,TIP,1.126767,0.005815,-0.224623,2023-10-01
4,DBC,1.037897,0.001809,-0.783860,2020-04-01
5,TLT,0.975901,-0.001184,-0.576096,2026-09-01
6,CASH,0.834796,-0.008734,-0.237810,2022-10-01


In [ ]:
conn.execute(""" 
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    ),

    cpi_filled AS (
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    inflation AS (
        SELECT 
            month,
            cpi,
            cpi_is_imputed,
            cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
            cpi / LAG(cpi,12) OVER (ORDER BY month) -1   AS yoy_inflation
        FROM cpi_filled 
    ),

    cash_returns AS (
        SELECT 
            date        AS month,
            'CASH'      AS ticker,
            POWER(1 + LAG(value) OVER (ORDER BY month) / 100, 1.0 /12)  - 1     AS nominal_return
        FROM stg_fred
        WHERE series_id = 'TB3MS'
    ),

    all_returns AS (
        SELECT month, ticker, nominal_return FROM monthly_returns
        UNION ALL
        SELECT month, ticker, nominal_return FROM cash_returns
    ),

    real_returns AS (
        SELECT
            a.month,
            ticker,
            nominal_return,
            monthly_inflation,
            yoy_inflation,
            (1 + nominal_return) / (1 + monthly_inflation) - 1   AS real_return
        FROM all_returns a
        INNER JOIN inflation b
        ON a.month = b.month
        WHERE a.month >= DATE '2006-03-01'
        AND a.month <= (SELECT MAX(month) FROM inflation WHERE monthly_inflation IS NOT NULL)
    ),

    wealth AS (
        SELECT
            month,
            ticker,
            nominal_return,
            real_return,
            EXP(SUM(LN(1 + nominal_return)) OVER (
                PARTITION BY ticker
                ORDER BY month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )) AS nominal_wealth,
            EXP(SUM(LN(1 + real_return)) OVER (
                PARTITION BY ticker
                ORDER BY month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )) AS real_wealth
        FROM real_returns
    ),

    drawdowns AS (
        SELECT
            month,
            ticker,
            nominal_return,
            real_return,
            nominal_wealth,
            real_wealth,
            real_wealth / GREATEST(1.0, MAX(real_wealth) OVER (
                PARTITION BY ticker
                ORDER BY month
            )) - 1 AS real_drawdown
        FROM wealth
    ),

    returns_by_regime AS (
        SELECT
            month,
            ticker,
            nominal_return,
            monthly_inflation,
            yoy_inflation,
            real_return,
            CASE
                WHEN yoy_inflation >= 0.04 THEN 'high'
                WHEN yoy_inflation > 0.02  THEN 'moderate'
                WHEN yoy_inflation <= 0.02 THEN 'low'
            END AS regime
        FROM real_returns
        )

        SELECT
            ticker,
            regime,
            COUNT(*) AS n_months,
            POWER(EXP(AVG(LN(1 + real_return))), 12) - 1 AS annualized_real_return,
            AVG(CASE WHEN real_return > 0 THEN 1 ELSE 0 END) AS share_positive
        FROM returns_by_regime 
        GROUP BY ticker, regime
        ORDER BY ticker, regime 
   """ ).df()




,ticker,regime,n_months,annualized_real_return,share_positive
0,CASH,high,38,-0.038284,0.289474
1,CASH,low,99,-0.005970,0.363636
2,CASH,moderate,110,-0.000824,0.472727
3,DBC,high,38,0.045935,0.552632
4,DBC,low,99,-0.008606,0.484848
5,DBC,moderate,110,-0.003671,0.527273
6,GLD,high,38,0.030261,0.500000
7,GLD,low,99,0.068020,0.525253
8,GLD,moderate,110,0.085943,0.536364
9,SPY,high,38,-0.104752,0.473684


In [ ]:
conn.execute(""" 
    WITH monthly_prices AS (
    SELECT 
        date_trunc('month', date)                                                                       AS month,
        ticker,
        date,
        adj_close
    FROM (
            SELECT
                date,
                ticker,
                adj_close,
                ROW_NUMBER() OVER (PARTITION BY ticker, date_trunc('month', date) ORDER BY date DESC)   AS rn
            FROM stg_prices
         ) AS sub   
    WHERE sub.rn = 1    
    ),

    monthly_returns AS (
            SELECT
                month,
                ticker,
                adj_close,
                adj_close / LAG(adj_close) OVER (PARTITION BY ticker ORDER BY month) - 1                AS nominal_return
            FROM monthly_prices
    ),

    cpi_filled AS (
        SELECT
            date            AS month, 
            value IS NULL   AS cpi_is_imputed,
            CASE
                WHEN value IS NULL THEN SQRT(LAG(value) OVER (ORDER BY month)* LEAD(value) OVER (ORDER BY month))
                ELSE value
            END AS cpi

        FROM stg_fred
        WHERE series_id = 'CPIAUCSL'
    ),

    inflation AS (
        SELECT 
            month,
            cpi,
            cpi_is_imputed,
            cpi / LAG(cpi) OVER (ORDER BY month) -1      AS monthly_inflation,
            cpi / LAG(cpi,12) OVER (ORDER BY month) -1   AS yoy_inflation
        FROM cpi_filled 
    ),

    cash_returns AS (
        SELECT 
            date        AS month,
            'CASH'      AS ticker,
            POWER(1 + LAG(value) OVER (ORDER BY month) / 100, 1.0 /12)  - 1     AS nominal_return
        FROM stg_fred
        WHERE series_id = 'TB3MS'
    ),

    all_returns AS (
        SELECT month, ticker, nominal_return FROM monthly_returns
        UNION ALL
        SELECT month, ticker, nominal_return FROM cash_returns
    ),

    real_returns AS (
        SELECT
            a.month,
            ticker,
            nominal_return,
            monthly_inflation,
            yoy_inflation,
            (1 + nominal_return) / (1 + monthly_inflation) - 1   AS real_return
        FROM all_returns a
        INNER JOIN inflation b
        ON a.month = b.month
        WHERE a.month >= DATE '2006-03-01'
        AND a.month <= (SELECT MAX(month) FROM inflation WHERE monthly_inflation IS NOT NULL)
    ),

    wealth AS (
        SELECT
            month,
            ticker,
            nominal_return,
            real_return,
            EXP(SUM(LN(1 + nominal_return)) OVER (
                PARTITION BY ticker
                ORDER BY month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )) AS nominal_wealth,
            EXP(SUM(LN(1 + real_return)) OVER (
                PARTITION BY ticker
                ORDER BY month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )) AS real_wealth
        FROM real_returns
    ),

    drawdowns AS (
        SELECT
            month,
            ticker,
            nominal_return,
            real_return,
            nominal_wealth,
            real_wealth,
            real_wealth / GREATEST(1.0, MAX(real_wealth) OVER (
                PARTITION BY ticker
                ORDER BY month
            )) - 1 AS real_drawdown
        FROM wealth
    ),

    returns_by_regime AS (
        SELECT
            month,
            ticker,
            nominal_return,
            monthly_inflation,
            yoy_inflation,
            real_return,
            CASE
                WHEN yoy_inflation >= 0.04 THEN 'high'
                WHEN yoy_inflation > 0.02  THEN 'moderate'
                WHEN yoy_inflation <= 0.02 THEN 'low'
            END AS regime
        FROM real_returns
        )

        SELECT
            month,
            regime,
        FROM returns_by_regime
        WHERE regime = 'high'

   """ ).df()

,month,regime
0,2006-06-01,high
1,2006-07-01,high
2,2007-11-01,high
3,2007-12-01,high
4,2008-01-01,high
...,...,...
261,2023-02-01,high
262,2023-03-01,high
263,2023-04-01,high
264,2023-05-01,high
